# Structured Output

**Module:** 07-prompt-engineering

**Notebook:** `05-structured-output.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Why Structured Output?** with clear contracts and failure modes
- Explain and apply **JSON Outputs** with clear contracts and failure modes
- Explain and apply **JSON Schema** with clear contracts and failure modes
- Explain and apply **Pydantic** with clear contracts and failure modes
- Explain and apply **Structured Extraction** with clear contracts and failure modes
- Explain and apply **Function Schema** with clear contracts and failure modes
- Explain and apply **Validation & Repair** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Structured Output

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Why Structured Output?**
2. **JSON Outputs**
3. **JSON Schema**
4. **Pydantic**
5. **Structured Extraction**
6. **Function Schema**
7. **Validation & Repair**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Why Structured Output?

### Definition
**Why Structured Output?** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Why Structured Output? typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Why Structured Output?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Why Structured Output? as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Why Structured Output? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Why Structured Output?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Why Structured Output? when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Why Structured Output? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Why Structured Output?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Why Structured Output?"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


## JSON Outputs

### Definition
**JSON Outputs** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around JSON Outputs typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For JSON Outputs: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain JSON Outputs as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating JSON Outputs as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for JSON Outputs
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use JSON Outputs when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "JSON Outputs" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "JSON Outputs"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


### Worked scenario — JSON Outputs

**Situation:** A team wants to productionize a feature involving **JSON Outputs**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## JSON Schema

### Definition
**JSON Schema** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around JSON Schema typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For JSON Schema: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain JSON Schema as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating JSON Schema as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for JSON Schema
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use JSON Schema when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "JSON Schema" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "JSON Schema"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


## Pydantic

### Definition
**Pydantic** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Pydantic typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Pydantic: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Pydantic as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Pydantic as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Pydantic
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Pydantic when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Pydantic" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Pydantic"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


### Worked scenario — Pydantic

**Situation:** A team wants to productionize a feature involving **Pydantic**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Structured Extraction

### Definition
**Structured Extraction** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Structured Extraction typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Structured Extraction: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Structured Extraction as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Structured Extraction as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Structured Extraction
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Structured Extraction when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Structured Extraction" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Structured Extraction"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


## Function Schema

### Definition
**Function Schema** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Function Schema typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Function Schema: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Function Schema as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Function Schema as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Function Schema
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Function Schema when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Function Schema" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Function Schema"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


### Worked scenario — Function Schema

**Situation:** A team wants to productionize a feature involving **Function Schema**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Validation & Repair

### Definition
**Validation & Repair** is a core building block in 05-structured-output within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Validation & Repair typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Validation & Repair: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Validation & Repair as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Validation & Repair as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Validation & Repair
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Validation & Repair when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Validation & Repair" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Validation & Repair"
    notebook: str = "05-structured-output"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


## Comparison Snapshot

Use this table when reviewing designs in **Structured Output**.

| Topic | Do | Don't |
|-------|----|-------|
| Why Structured Output? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| JSON Outputs | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| JSON Schema | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Pydantic | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Structured Extraction | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Function Schema | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Why Structured Output? | Key concept covered in this notebook; see its section for definition and pitfalls |
| JSON Outputs | Key concept covered in this notebook; see its section for definition and pitfalls |
| JSON Schema | Key concept covered in this notebook; see its section for definition and pitfalls |
| Pydantic | Key concept covered in this notebook; see its section for definition and pitfalls |
| Structured Extraction | Key concept covered in this notebook; see its section for definition and pitfalls |
| Function Schema | Key concept covered in this notebook; see its section for definition and pitfalls |
| Validation & Repair | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Structured Output** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **07-prompt-engineering**.


## Try It Yourself

1. Implement a failing test/fixture for **Why Structured Output?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **JSON Outputs**, then fix your demo until it passes.
3. Implement a failing test/fixture for **JSON Schema**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Pydantic**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Structured Extraction**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
